# The whole brain: running, breaking and calibrating it

In notebook 02 we made a tiny network fire. Here we run **all 166,700 neurons** of the male fly's nervous system, and discover that the raw connectome, simulated with simple neuron rules, **runs out of control**. Most of this notebook is the detective story of finding out why, and fixing it.

| Part | What happens |
|---|---|
| A. First run | We smell a pheromone. The whole brain explodes into maximum activity. |
| B. Scaling synapses | Male synapse counts run high. Scaling them down helps a lot, but not enough. |
| C. Dose-response | A tiny smell causes the same explosion as a huge one: the brain is self-sustaining. |
| D. Hunting loops | We find circuits that excite themselves in a loop, and cut them one by one. |
| E. A better simulator and a general fix | We correct the simulator and add one biological rule that stops all the loops. |
| F. Can smell steer the fly? | We look for neurons that could turn the fly toward food. |

At the end, an epilogue explains what happened next, outside this notebook.

# Part A: The first whole-brain run

## A1. Load the network

We load the two files saved in notebook 01: the neuron list and the signed connection matrix `W`.

The safety check matters: row *i* of the table must describe neuron *i* of the matrix. If they ever got out of order, every result would be attached to the wrong neuron.

In [ ]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp

neurons = pd.read_parquet("../data/neurons.parquet")
W = sp.load_npz("../data/W.npz")
print(neurons.shape, W.shape)
print("Row labels match positions:", (neurons.index == np.arange(len(neurons))).all())

## A2. Pick an experiment: a pheromone

We need a pathway where we know what *should* happen, so we can check the simulation.

**How flies smell:**
1. **Olfactory receptor neurons (ORNs)** sit in the antenna. Each ORN type carries one kind of receptor and detects certain odors.
2. All ORNs of one type send their fibers to the same small ball of brain tissue, a **glomerulus**, in the **antennal lobe**. There are about 50 glomeruli, named with codes like `DA1`.
3. **Projection neurons (PNs)** in each glomerulus carry that smell onward, deeper into the brain.

The **DA1** glomerulus detects **cVA**, a pheromone males leave on females during mating. If the simulation works, stimulating DA1 ORNs should mainly activate **DA1 PNs**.

The search finds 204 DA1 ORNs and 15 DA1 PNs (`lPN` and `vPN` are two PN families).

In [ ]:
def find_types(text):
    mask = neurons["type"].str.contains(text, case=False, na=False)
    return neurons[mask].groupby(["type", "superclass"]).size()

find_types("DA1")

## A3. Parameters

Same values as notebook 02, from Shiu et al. (Nature 2024). Each notebook runs separately, so we define them again.

In [ ]:
dt = 0.1
v_rest, v_reset, v_th = -52.0, -52.0, -45.0
tau_m, tau_syn = 20.0, 5.0
t_ref, delay = 2.2, 1.8
w_syn = 0.275
stim_rate = 150.0
stim_strength = 250 * w_syn

## A4. The whole-brain simulator (version 1)

The same rules as `simulate_network` in notebook 02, with two changes for size:

- **The sparse matrix.** `W_mv[fired]` picks the outgoing connections of the neurons that just fired. With a sparse matrix, `.sum(axis=0)` returns a special 2D object, so `np.asarray(...).ravel()` turns it back into a plain list of numbers.
- **`float32`** numbers instead of `float64`: half the memory, faster, and still far more precise than we need.

It prints progress every 100 ms of simulated time, including how many neurons fired in that instant: a quick view of how activity spreads.

In [ ]:
def simulate_brain(W, stim, duration_ms=1000, seed=0):
    n = W.shape[0]
    steps = int(duration_ms / dt)
    delay_steps = int(delay / dt)
    ref_steps = int(t_ref / dt)
    rng = np.random.default_rng(seed)

    W_mv = (W * w_syn).astype(np.float32).tocsr()
    v = np.full(n, v_rest, dtype=np.float32)
    g = np.zeros(n, dtype=np.float32)
    ref = np.zeros(n, dtype=np.int32)
    buffer = np.zeros((delay_steps, n), dtype=np.float32)
    spike_counts = np.zeros(n, dtype=np.int32)

    start = time.time()
    for t in range(steps):
        slot = t % delay_steps
        g += buffer[slot]
        buffer[slot] = 0

        hits = stim[rng.random(len(stim)) < stim_rate * dt / 1000]
        g[hits] += stim_strength

        v += dt * (v_rest - v + g) / tau_m
        g -= dt * g / tau_syn

        refractory = ref > 0
        v[refractory] = v_reset
        ref[refractory] -= 1

        fired = np.flatnonzero((v >= v_th) & ~refractory)
        v[fired] = v_reset
        ref[fired] = ref_steps
        spike_counts[fired] += 1

        if len(fired) > 0:
            buffer[slot] += np.asarray(W_mv[fired].sum(axis=0)).ravel()

        if t % 1000 == 0:
            print(f"{t * dt:6.0f} ms simulated, {time.time() - start:5.1f} s elapsed, "
                  f"{len(fired)} neurons firing now", flush=True)

    return spike_counts / (duration_ms / 1000)

## A5. Run it: stimulate all 204 DA1 ORNs for one second

**What happened:** about **300–370 neurons fire in every 0.1 ms step**. That's a lot. In a calm brain, it would be a handful.

In [ ]:
stim = np.flatnonzero(neurons["type"] == "ORN_DA1")
print(f"Stimulating {len(stim)} neurons")

rates = simulate_brain(W, stim, duration_ms=1000)

## A6. Who responded?

We list the most active cell types, excluding the stimulated ORNs.

**Result: about 25,700 neurons fired, and the top types all fire at ~430 Hz.**

**Why 430 Hz is a warning sign.** That's essentially the **fastest a neuron can fire** in this model: after each spike it waits 2.2 ms (refractory), then needs a moment to recharge, so about 2.3 ms per spike, or 1000 ÷ 2.3 ≈ 435 spikes per second. Neurons stuck at their maximum, nonstop, is **runaway excitation**, the simulation's version of a seizure.

The list is full of antennal lobe local neurons (`lLN...`) and PNs from *other* glomeruli (`DC1`, `VA6`, `DM1`). The pheromone signal didn't stay in its own pathway: it spread everywhere.

In [ ]:
results = neurons[["bodyId", "type", "superclass", "consensus_nt"]].copy()
results["rate_hz"] = rates

responders = results[(results["rate_hz"] > 0) & ~results.index.isin(stim)]
print(f"{len(responders)} neurons fired (not counting the stimulated ones)")

(responders.groupby(["type", "superclass"])["rate_hz"]
           .agg(["count", "mean"])
           .sort_values("mean", ascending=False)
           .head(25))

## A7. How bad is it?

**About 25,900 neurons fired, 310 of them near the ceiling.** All 15 DA1 PNs sit at 369–414 Hz. And the activity reached the nerve cord (`vnc_intrinsic`), the optic lobes, even descending and motor neurons.

In [ ]:
firing = results[results["rate_hz"] > 0]
print(f"Neurons firing: {len(firing):,} of {len(results):,}")
print(f"Neurons near the ceiling (>400 Hz): {(results['rate_hz'] > 400).sum():,}")

print("\nDA1 projection neurons:")
print(results[results["type"].str.startswith("DA1_", na=False)][["type", "rate_hz"]])

print("\nFiring neurons by superclass:")
firing.groupby("superclass").size().sort_values(ascending=False)

# Part B: The synapse-count problem

## B1. Add a weight scale

**Diagnosis:** the model's synapse strength (0.275 mV) was tuned by Shiu et al. on the **female FlyWire** connectome. The MaleCNS team found that **male synapse counts run about 1.7× higher** for the same connections, probably from differences in imaging and synapse detection, not biology. They multiply male weights by **0.581** when comparing the two datasets.

So every connection in our model was 1.7× too strong. We add a `weight_scale` option. With the default of 1.0, earlier results are unchanged.

In [ ]:
def simulate_brain(W, stim, duration_ms=1000, seed=0, weight_scale=1.0):   # changed
    n = W.shape[0]
    steps = int(duration_ms / dt)
    delay_steps = int(delay / dt)
    ref_steps = int(t_ref / dt)
    rng = np.random.default_rng(seed)

    W_mv = (W * (w_syn * weight_scale)).astype(np.float32).tocsr()          # changed
    v = np.full(n, v_rest, dtype=np.float32)
    g = np.zeros(n, dtype=np.float32)
    ref = np.zeros(n, dtype=np.int32)
    buffer = np.zeros((delay_steps, n), dtype=np.float32)
    spike_counts = np.zeros(n, dtype=np.int32)

    start = time.time()
    for t in range(steps):
        slot = t % delay_steps
        g += buffer[slot]
        buffer[slot] = 0

        hits = stim[rng.random(len(stim)) < stim_rate * dt / 1000]
        g[hits] += stim_strength

        v += dt * (v_rest - v + g) / tau_m
        g -= dt * g / tau_syn

        refractory = ref > 0
        v[refractory] = v_reset
        ref[refractory] -= 1

        fired = np.flatnonzero((v >= v_th) & ~refractory)
        v[fired] = v_reset
        ref[fired] = ref_steps
        spike_counts[fired] += 1

        if len(fired) > 0:
            buffer[slot] += np.asarray(W_mv[fired].sum(axis=0)).ravel()

        if t % 1000 == 0:
            print(f"{t * dt:6.0f} ms simulated, {time.time() - start:5.1f} s elapsed, "
                  f"{len(fired)} neurons firing now", flush=True)

    return spike_counts / (duration_ms / 1000)

## B2. Rerun with the 0.581 correction

**Much better:** only 81 neurons near the ceiling, and the progress lines level off around 180–200 neurons per step instead of growing. But 16,774 neurons still fired, and DA1 PNs at 310–390 Hz are far faster than real ones.

In [ ]:
rates = simulate_brain(W, stim, duration_ms=300, weight_scale=0.581)

results["rate_hz"] = rates
firing = results[results["rate_hz"] > 0]
print(f"Neurons firing: {len(firing):,}")
print(f"Near the ceiling: {(results['rate_hz'] > 400).sum():,}")
print(results[results["type"].str.startswith("DA1_", na=False)][["type", "rate_hz"]])

## B3. How strongly did neurons respond?

`pd.cut` sorts firing rates into bands, like grades.

**90% of the nervous system stayed silent**, good. But **~10,000 neurons fired at 50 Hz or more**, and 3,296 at over 200 Hz. Smelling one pheromone shouldn't strongly drive 10,000 neurons.

In [ ]:
bins = [0, 1, 10, 50, 100, 200, 500]
labels = ["0", "1–10", "10–50", "50–100", "100–200", "200+"]
results["rate_band"] = pd.cut(results["rate_hz"], bins=bins, labels=labels, right=False)
results["rate_band"].value_counts().sort_index()

## B4. The strongest responders

**The whole antennal lobe lit up, not just DA1.** PNs of other glomeruli fire almost as fast as DA1's own. The simulation lost **selectivity**: it can't tell this smell apart from others.

The suspects are the **local neurons** (`lLN...`), which connect many glomeruli to each other. Many are labeled `acetylcholine`, so in the model they *spread* excitation sideways.

But some real biology also shows up further down the list: mushroom body output neurons (`MBON09`, `MBON11`), dopamine neurons (`PPL101`) and a descending neuron (`DNb05`). The wiring carries the signal through plausible pathways; the problem is how hard it's driven.

In [ ]:
strong = results[(results["rate_hz"] >= 50) & ~results.index.isin(stim)]
print(f"{len(strong)} neurons firing at 50 Hz or more")

(strong.groupby(["type", "superclass", "consensus_nt"])["rate_hz"]
       .agg(["count", "mean"])
       .sort_values("mean", ascending=False)
       .head(30))

# Part C: The dose-response experiment

## C1. Define the groups we'll measure

A **dose-response experiment** is a standard method: vary the stimulus strength and measure the response. We'll compare two groups:
- **DA1 PNs**: should respond to the pheromone.
- **Other PNs**: should mostly stay quiet.

`str.contains("PN")` plus `superclass == "cb_intrinsic"` finds projection neurons (an approximation good enough for comparing groups).

In [ ]:
is_pn = (results["type"].str.contains("PN", na=False) &
         (results["superclass"] == "cb_intrinsic")).to_numpy()
is_da1 = results["type"].str.startswith("DA1_", na=False).to_numpy()
is_other_pn = is_pn & ~is_da1

print("DA1 PNs:", is_da1.sum(), " Other PNs:", is_other_pn.sum())

## C2. Stimulate 5, 10, 25, 50, 100 and all 204 ORNs

**The table changes the diagnosis:**
- **Even 5 ORNs ignite about 8,900 neurons.** A 40× stronger stimulus barely changes that number. A normal network responds roughly in proportion to its input; this one acts like a **switch**.
- **Other PNs fire at ~130–160 Hz regardless of the stimulus**, sometimes faster than DA1's own PNs. Activity that doesn't depend on the input must be coming from **inside the network**.
- DA1 PNs *do* respond in a graded way (72 → 361 Hz), so the direct pathway works.

The network is **self-sustaining**: once kicked, it keeps itself going.

In [ ]:
all_orns = np.flatnonzero(neurons["type"] == "ORN_DA1")
pick = np.random.default_rng(42)

rows = []
for k in [5, 10, 25, 50, 100, 204]:
    stim_k = pick.choice(all_orns, size=k, replace=False)
    r = simulate_brain(W, stim_k, duration_ms=300, weight_scale=0.581)
    rows.append({
        "ORNs stimulated": k,
        "DA1 PN rate": r[is_da1].mean(),
        "other PN rate": r[is_other_pn].mean(),
        "neurons ≥50 Hz": int((r >= 50).sum()) - k,
    })

dose = pd.DataFrame(rows)
dose

# Part D: Hunting for loops

## D1. Suspect 1: neurons that excite themselves

A neuron with a strong excitatory synapse onto **itself** (an **autapse**) could fire forever: each spike re-triggers it 1.8 ms later.

`W.diagonal()` gives the matrix cells where sender = receiver.

**Result:** 101 neurons have self-connections, but the strongest has only **29 synapses** (a motor neuron, `MN11V`). With our scaled weights that's a bump of under 1 mV, far below the 7 mV needed to fire. Too weak to matter. Suspect cleared.

In [ ]:
self_w = W.diagonal()
print("Neurons with self-connections:", (self_w != 0).sum())
print("  excitatory:", (self_w > 0).sum(), "  inhibitory:", (self_w < 0).sum())

(neurons.assign(self_synapses=self_w)
        .sort_values("self_synapses", ascending=False)
        [["type", "superclass", "consensus_nt", "self_synapses"]]
        .head(15))

## D2. Simulator version 3: switching the stimulus off

> **Note about this cell.** In the original notebook, this cell was later overwritten with the corrected simulator. The results in Part D were produced by *this* version, so it's restored here so the notebook reruns faithfully. The corrected version appears in Part E.

New options:
- **`stim_off_ms`** stops the stimulus partway through. This is the key test: **if activity continues after the input stops, the network is sustaining itself.**
- **`activity`** records how many neurons fired at every step, so we can plot activity over time.
- **`remove_self_connections`** deletes autapses, to test suspect 1 directly.
- It now **returns two things**: `rates, activity = simulate_brain(...)`.

In [ ]:
def simulate_brain(W, stim, duration_ms=1000, seed=0, weight_scale=1.0,
                   stim_off_ms=None, remove_self_connections=False, verbose=True):
    n = W.shape[0]
    steps = int(duration_ms / dt)
    delay_steps = int(delay / dt)
    ref_steps = int(t_ref / dt)
    rng = np.random.default_rng(seed)

    W_mv = (W * (w_syn * weight_scale)).astype(np.float32).tocsr()
    if remove_self_connections:
        W_mv = (W_mv - sp.diags(W_mv.diagonal())).tocsr()
        W_mv.eliminate_zeros()

    v = np.full(n, v_rest, dtype=np.float32)
    g = np.zeros(n, dtype=np.float32)
    ref = np.zeros(n, dtype=np.int32)
    buffer = np.zeros((delay_steps, n), dtype=np.float32)
    spike_counts = np.zeros(n, dtype=np.int32)
    activity = np.zeros(steps, dtype=np.int32)

    start = time.time()
    for t in range(steps):
        slot = t % delay_steps
        g += buffer[slot]
        buffer[slot] = 0

        if stim_off_ms is None or t * dt < stim_off_ms:
            hits = stim[rng.random(len(stim)) < stim_rate * dt / 1000]
            g[hits] += stim_strength

        v += dt * (v_rest - v + g) / tau_m
        g -= dt * g / tau_syn

        refractory = ref > 0
        v[refractory] = v_reset
        ref[refractory] -= 1

        fired = np.flatnonzero((v >= v_th) & ~refractory)
        v[fired] = v_reset
        ref[fired] = ref_steps
        spike_counts[fired] += 1
        activity[t] = len(fired)

        if len(fired) > 0:
            buffer[slot] += np.asarray(W_mv[fired].sum(axis=0)).ravel()

        if verbose and t % 1000 == 0:
            print(f"{t * dt:6.0f} ms simulated, {time.time() - start:5.1f} s elapsed, "
                  f"{len(fired)} neurons firing now", flush=True)

    rates = spike_counts / (duration_ms / 1000)
    return rates, activity

## D3. The switch-off test

Stimulate 5 ORNs for 100 ms, then nothing until 400 ms. Plot spikes per millisecond on a **log scale** (each step up is ×10).

**Three phases appear:**
1. **0–50 ms, smoldering:** a few spikes.
2. **50–100 ms, ignition:** a straight climb on a log scale means **exponential growth**: each wave of spikes triggers more than the last, a chain reaction.
3. **After 100 ms, self-sustaining:** ~2,000 spikes per ms with **no input at all**.

The two lines overlap almost exactly, so autapses aren't the cause.

A state the network falls into and can't leave on its own is called an **attractor**. That explains the dose-response puzzle: any stimulus strong enough to cross the tipping point drops the brain into the same attractor.

In [ ]:
import matplotlib.pyplot as plt

stim_5 = np.random.default_rng(42).choice(all_orns, size=5, replace=False)

runs = {
    "original": dict(),
    "self-connections removed": dict(remove_self_connections=True),
}

plt.figure(figsize=(10, 4))
for label, options in runs.items():
    rates, activity = simulate_brain(W, stim_5, duration_ms=400, weight_scale=0.581,
                                     stim_off_ms=100, verbose=False, **options)
    spikes_per_ms = activity.reshape(-1, 10).sum(axis=1)
    plt.plot(spikes_per_ms, label=label, linewidth=0.8)
    print(f"{label}: {(rates >= 50).sum():,} neurons ≥50 Hz, "
          f"spikes in last 100 ms: {spikes_per_ms[300:].sum():,}")

plt.axvline(100, color="red", linestyle="--", label="stimulus off")
plt.xlabel("Time (ms)")
plt.ylabel("Spikes per ms (whole nervous system)")
plt.yscale("log")
plt.legend()
plt.show()

## D4. Who keeps the activity going?

Most of the 400 ms came after the stimulus stopped, so these rates mostly reflect the self-sustained state.

**9,210 neurons**, almost all in the central brain (`cb_intrinsic`): one or a few specific circuits, not the whole nervous system.

In [ ]:
results["rate_hz"] = rates
core = results[results["rate_hz"] >= 50]
print(f"{len(core):,} neurons in the self-sustaining state\n")

print(core.groupby("superclass").size().sort_values(ascending=False).head(10))
print()
print(core.groupby("consensus_nt").size().sort_values(ascending=False))

## D5. Is there a weight scale that works?

Turn all connections down and look for the tipping point.

**Result: a cliff.** At 0.5 the brain explodes; at 0.4 almost nothing gets through (DA1 PNs at 0.5 Hz). **No single global scale gives a healthy brain.** Scaling everything equally is too blunt; we must fix the specific circuits misbehaving.

In [ ]:
rows = []
for scale in [0.581, 0.5, 0.4, 0.3, 0.2]:
    r, act = simulate_brain(W, stim_5, duration_ms=400, weight_scale=scale,
                            stim_off_ms=100, verbose=False)
    per_ms = act.reshape(-1, 10).sum(axis=1)
    rows.append({
        "weight_scale": scale,
        "peak spikes/ms": int(per_ms.max()),
        "spikes after stimulus (last 100 ms)": int(per_ms[300:].sum()),
        "DA1 PN rate": r[is_da1].mean(),
    })
    print(f"scale {scale} done", flush=True)

pd.DataFrame(rows)

## D6. Suspect 2: dopamine and the mushroom body

**All 4,064 Kenyon cells** and **362 of 392 dopamine neurons** are in the runaway core.

**Background:** the **mushroom body** is the fly's learning center. Its ~4,000 **Kenyon cells** (cholinergic, so excitatory) represent smells. **Dopamine neurons** teach it which smells are good or bad, and the two are heavily interconnected.

**The model's mistake:** real dopamine doesn't make neurons fire directly. It acts slowly, adjusting how connections behave over seconds to minutes. Our model treats it as fast excitation, creating a loop that doesn't exist in reality:
Kenyon cells → excite dopamine neurons → "excite" Kenyon cells → ...

In [ ]:
is_kc = results["type"].str.startswith("KC", na=False)
print(f"Kenyon cells: {(is_kc & (results['rate_hz'] >= 50)).sum():,} "
      f"in core, of {is_kc.sum():,} total\n")

for transmitter in ["dopamine", "octopamine", "serotonin"]:
    total = (neurons["consensus_nt"] == transmitter).sum()
    in_core = (core["consensus_nt"] == transmitter).sum()
    print(f"{transmitter:12} {in_core:4} of {total:4} in core")

print("\nMost common types in the core:")
core["type"].value_counts().head(15)

## D7. Silence the modulators' fast effects

We zero the **outgoing** connections of dopamine, octopamine and serotonin neurons.

**The matrix trick:** multiplying by a diagonal matrix *from the left* scales each **row**. Rows of normal neurons are ×1, rows of modulators ×0. Since rows are outgoing connections, modulators can still receive input and fire, but can't affect anyone.

In [ ]:
modulators = ["dopamine", "octopamine", "serotonin"]
is_modulator = neurons["consensus_nt"].isin(modulators).to_numpy()
print("Modulatory neurons:", is_modulator.sum())

row_keep = sp.diags((~is_modulator).astype(np.float32))
W_nomod = (row_keep @ W).tocsr()
print("Connections before:", W.nnz, " after:", W_nomod.count_nonzero())

## D8. Retest

**A small improvement** (peak 2,145 → 1,462 spikes/ms), but it still sustains itself (135,438 spikes after the stimulus). Dopamine wasn't the main loop.

In [ ]:
r, act = simulate_brain(W_nomod, stim_5, duration_ms=400, weight_scale=0.581,
                        stim_off_ms=100, verbose=False)
per_ms = act.reshape(-1, 10).sum(axis=1)
print(f"Peak spikes/ms: {per_ms.max():,}")
print(f"Spikes after stimulus (last 100 ms): {per_ms[300:].sum():,}")

rows = []
for k in [5, 25, 100, 204]:
    stim_k = np.random.default_rng(42).choice(all_orns, size=k, replace=False)
    r, _ = simulate_brain(W_nomod, stim_k, duration_ms=300, weight_scale=0.581, verbose=False)
    rows.append({
        "ORNs stimulated": k,
        "DA1 PN rate": r[is_da1].mean(),
        "other PN rate": r[is_other_pn].mean(),
        "neurons ≥50 Hz": int((r >= 50).sum()) - k,
    })
pd.DataFrame(rows)

## D9. Suspect 3: Kenyon cells exciting each other

**51% of all Kenyon cell output synapses go onto other Kenyon cells.**

In real flies, many of these are **axo-axonal synapses**: from one neuron's axon onto *another's axon*, not onto the dendrites where neurons normally collect input. They adjust signals locally rather than making the cell fire.

**Our model can't tell the difference.** It treats each neuron as a single point (a **point neuron**), so every synapse counts as full input. 4,000 excitatory Kenyon cells then look like a giant web feeding itself.

In [ ]:
is_kc_np = is_kc.to_numpy()
W_abs = abs(W)

kc_out = W_abs[is_kc_np]
kc_to_kc = kc_out[:, is_kc_np].sum()
kc_to_all = kc_out.sum()
print(f"Kenyon cell output synapses: {kc_to_all:,.0f}")
print(f"  onto other Kenyon cells:   {kc_to_kc:,.0f}  ({100 * kc_to_kc / kc_to_all:.0f}%)")

## D10. Remove Kenyon-cell-to-Kenyon-cell connections

We filter the connection list in COO format (sender, receiver, value): keep every connection that isn't KC→KC.

**Result:** core shrinks again (peak 982 spikes/ms), but activity still persists (87,427). Another loop remains.

In [ ]:
coo = W_nomod.tocoo()
kc_pair = is_kc_np[coo.row] & is_kc_np[coo.col]
W_fixed = sp.csr_matrix((coo.data[~kc_pair], (coo.row[~kc_pair], coo.col[~kc_pair])),
                        shape=W.shape)
print("Removed KC→KC connections:", kc_pair.sum())

r, act = simulate_brain(W_fixed, stim_5, duration_ms=400, weight_scale=0.581,
                        stim_off_ms=100, verbose=False)
per_ms = act.reshape(-1, 10).sum(axis=1)
print(f"Peak spikes/ms: {per_ms.max():,}")
print(f"Spikes after stimulus (last 100 ms): {per_ms[300:].sum():,}")

rows = []
for k in [5, 25, 100, 204]:
    stim_k = np.random.default_rng(42).choice(all_orns, size=k, replace=False)
    r, _ = simulate_brain(W_fixed, stim_k, duration_ms=300, weight_scale=0.581, verbose=False)
    rows.append({
        "ORNs stimulated": k,
        "DA1 PN rate": r[is_da1].mean(),
        "other PN rate": r[is_other_pn].mean(),
        "neurons ≥50 Hz": int((r >= 50).sum()) - k,
    })
pd.DataFrame(rows)

# Part E: A better simulator and a general fix

## E1. Correcting the simulator

Comparing our code line by line with Shiu et al.'s original model revealed **three differences**:

1. **Clearing input after a spike:** firing now also resets the neuron's synaptic input `g` to 0. Before, leftover input could make it fire again right away, exactly the self-feeding that sustains loops.
2. **Freezing during the refractory period:** a refractory neuron's voltage and input don't change (`awake` mask).
3. **Stimulus as a voltage kick:** stimulated neurons get a direct ~69 mV kick and have no refractory period, so each input event gives exactly one spike.

It also adds `stim_rate`, so experiments can use different smell strengths.

In [ ]:
stim_kick = 250 * w_syn

def simulate_brain(W, stim, duration_ms=1000, seed=0, weight_scale=1.0, stim_rate=stim_rate,
                   stim_off_ms=None, remove_self_connections=False, verbose=True):
    n = W.shape[0]
    steps = int(duration_ms / dt)
    delay_steps = int(delay / dt)
    ref_steps = int(t_ref / dt)
    rng = np.random.default_rng(seed)

    W_mv = (W * (w_syn * weight_scale)).astype(np.float32).tocsr()
    if remove_self_connections:
        W_mv = (W_mv - sp.diags(W_mv.diagonal())).tocsr()
        W_mv.eliminate_zeros()

    v = np.full(n, v_rest, dtype=np.float32)
    g = np.zeros(n, dtype=np.float32)
    ref = np.zeros(n, dtype=np.int32)
    buffer = np.zeros((delay_steps, n), dtype=np.float32)
    spike_counts = np.zeros(n, dtype=np.int32)
    activity = np.zeros(steps, dtype=np.int32)

    gets_refractory = np.ones(n, dtype=bool)
    gets_refractory[stim] = False

    start = time.time()
    for t in range(steps):
        slot = t % delay_steps
        g += buffer[slot]
        buffer[slot] = 0

        if stim_off_ms is None or t * dt < stim_off_ms:
            hits = stim[rng.random(len(stim)) < stim_rate * dt / 1000]
            v[hits] += stim_kick

        awake = ref == 0
        v[awake] += dt * (v_rest - v[awake] + g[awake]) / tau_m
        g[awake] -= dt * g[awake] / tau_syn
        ref[~awake] -= 1

        fired = np.flatnonzero((v > v_th) & awake)
        v[fired] = v_reset
        g[fired] = 0
        ref[fired[gets_refractory[fired]]] = ref_steps

        spike_counts[fired] += 1
        activity[t] = len(fired)

        if len(fired) > 0:
            buffer[slot] += np.asarray(W_mv[fired].sum(axis=0)).ravel()

        if verbose and t % 1000 == 0:
            print(f"{t * dt:6.0f} ms simulated, {time.time() - start:5.1f} s elapsed, "
                  f"{len(fired)} neurons firing now", flush=True)

    rates = spike_counts / (duration_ms / 1000)
    return rates, activity

## E2. Retest with the corrected simulator

**The bug fixes roughly halved the runaway:** original matrix 2,145 → 1,228 peak spikes/ms; `W_fixed` 982 → 564. But both still sustain themselves.

In [ ]:
tests = {
    "original W": W,
    "W_fixed (no modulators, no KC→KC)": W_fixed,
}

for label, matrix in tests.items():
    r, act = simulate_brain(matrix, stim_5, duration_ms=400, weight_scale=0.581,
                            stim_off_ms=100, verbose=False)
    per_ms = act.reshape(-1, 10).sum(axis=1)
    print(f"{label}:")
    print(f"   peak spikes/ms: {per_ms.max():,}")
    print(f"   spikes after stimulus: {per_ms[300:].sum():,}")
    print(f"   DA1 PN rate: {r[is_da1].mean():.1f}   other PN rate: {r[is_other_pn].mean():.1f}")

## E3. Who keeps it going now?

**A clue:** `HRN_VP1d` and `HRN_VP5` are **humidity and temperature sensory neurons**, still firing with no stimulus. Sensory neurons should only fire when the outside world stimulates them.

In the antennal lobe, brain neurons make many synapses back onto the sensory neurons' **axon terminals**. In real flies these only adjust how much signal the sensory neuron releases. The model, again, turns them into full excitation, letting the brain drive its own sensors in a loop.

In [ ]:
r_core, _ = simulate_brain(W_fixed, stim_5, duration_ms=400, weight_scale=0.581,
                           stim_off_ms=100, verbose=False)
results["rate_hz"] = r_core
core = results[results["rate_hz"] >= 50]
print(f"{len(core):,} neurons in the core\n")
print(core.groupby("superclass").size().sort_values(ascending=False).head(8))
print()
core["type"].value_counts().head(20)

## E4. Weight sweep with the corrected simulator

**The cliff is gone:** the response now changes smoothly, and DA1 PNs beat other PNs at every scale. But a loop persists even at 0.35, so it involves very strong connections.

In [ ]:
stim_25 = np.random.default_rng(42).choice(all_orns, size=25, replace=False)

rows = []
for scale in [0.581, 0.5, 0.45, 0.4, 0.35]:
    r_on, _ = simulate_brain(W_fixed, stim_25, duration_ms=300, weight_scale=scale, verbose=False)
    _, act = simulate_brain(W_fixed, stim_25, duration_ms=400, weight_scale=scale,
                            stim_off_ms=100, verbose=False)
    per_ms = act.reshape(-1, 10).sum(axis=1)
    rows.append({
        "weight_scale": scale,
        "spikes after stimulus": int(per_ms[300:].sum()),
        "DA1 PN rate": r_on[is_da1].mean(),
        "other PN rate": r_on[is_other_pn].mean(),
        "neurons ≥50 Hz": int((r_on >= 50).sum()) - len(stim_25),
    })
    print(f"scale {scale} done", flush=True)

pd.DataFrame(rows)

## E5. Rule: sensory neurons receive no input from the brain

Multiplying by a diagonal matrix *from the right* scales **columns** (incoming connections). Sensory neurons' columns become 0: they keep their outputs, but only the outside world drives them.

We remove 1.9 million synapses onto 17,937 sensory neurons.

In [ ]:
is_sensory = results["superclass"].str.contains("sensory", na=False).to_numpy()
print("Sensory neurons:", is_sensory.sum())

incoming = abs(W_fixed)[:, is_sensory].sum()
print(f"Synapses onto sensory neurons being removed: {incoming:,.0f}")

col_keep = sp.diags((~is_sensory).astype(np.float32))
W_sens = (W_fixed @ col_keep).tocsr()
W_sens.eliminate_zeros()
print("Connections left:", W_sens.nnz)

## E6. Retest

**Small improvement** (about 49,000 → 44,000 spikes after the stimulus). A real contributor, but not the main loop.

Guessing loops one at a time explains only pieces. Next, a **systematic** method.

In [ ]:
rows = []
for scale in [0.581, 0.5, 0.45]:
    r_on, _ = simulate_brain(W_sens, stim_25, duration_ms=300, weight_scale=scale, verbose=False)
    _, act = simulate_brain(W_sens, stim_25, duration_ms=400, weight_scale=scale,
                            stim_off_ms=100, verbose=False)
    per_ms = act.reshape(-1, 10).sum(axis=1)
    rows.append({
        "weight_scale": scale,
        "spikes after stimulus": int(per_ms[300:].sum()),
        "DA1 PN rate": r_on[is_da1].mean(),
        "other PN rate": r_on[is_other_pn].mean(),
        "neurons ≥50 Hz": int((r_on >= 50).sum()) - len(stim_25),
    })
    print(f"scale {scale} done", flush=True)

pd.DataFrame(rows)

## E7. Simulator version 5: count only late spikes

`count_after_ms` counts spikes only after a chosen time. Anything firing 100+ ms after the stimulus stopped is sustaining itself, so this isolates the loop's members.

In [ ]:
stim_kick = 250 * w_syn

def simulate_brain(W, stim, duration_ms=1000, seed=0, weight_scale=1.0, stim_rate=stim_rate,
                   stim_off_ms=None, remove_self_connections=False, verbose=True, count_after_ms=0):
    n = W.shape[0]
    steps = int(duration_ms / dt)
    delay_steps = int(delay / dt)
    ref_steps = int(t_ref / dt)
    rng = np.random.default_rng(seed)

    W_mv = (W * (w_syn * weight_scale)).astype(np.float32).tocsr()
    if remove_self_connections:
        W_mv = (W_mv - sp.diags(W_mv.diagonal())).tocsr()
        W_mv.eliminate_zeros()

    v = np.full(n, v_rest, dtype=np.float32)
    g = np.zeros(n, dtype=np.float32)
    ref = np.zeros(n, dtype=np.int32)
    buffer = np.zeros((delay_steps, n), dtype=np.float32)
    spike_counts = np.zeros(n, dtype=np.int32)
    activity = np.zeros(steps, dtype=np.int32)

    gets_refractory = np.ones(n, dtype=bool)
    gets_refractory[stim] = False

    start = time.time()
    for t in range(steps):
        slot = t % delay_steps
        g += buffer[slot]
        buffer[slot] = 0

        if stim_off_ms is None or t * dt < stim_off_ms:
            hits = stim[rng.random(len(stim)) < stim_rate * dt / 1000]
            v[hits] += stim_kick

        awake = ref == 0
        v[awake] += dt * (v_rest - v[awake] + g[awake]) / tau_m
        g[awake] -= dt * g[awake] / tau_syn
        ref[~awake] -= 1

        fired = np.flatnonzero((v > v_th) & awake)
        v[fired] = v_reset
        g[fired] = 0
        ref[fired[gets_refractory[fired]]] = ref_steps

        if t * dt >= count_after_ms:
            spike_counts[fired] += 1
        activity[t] = len(fired)

        if len(fired) > 0:
            buffer[slot] += np.asarray(W_mv[fired].sum(axis=0)).ravel()

        if verbose and t % 1000 == 0:
            print(f"{t * dt:6.0f} ms simulated, {time.time() - start:5.1f} s elapsed, "
                  f"{len(fired)} neurons firing now", flush=True)

    rates = spike_counts / ((duration_ms - count_after_ms) / 1000)
    return rates, activity

## E8. Map the loop directly

1. Find every neuron still firing late.
2. Cut out the part of the matrix where **both** sender and receiver are late-firing: the loop must live there.
3. Sum excitatory synapses by (sender type, receiver type).

**The top line is the culprit:** `lLN1_bc → lLN1_bc`, **86,864 synapses among just 30 neurons**, about 2,900 per neuron. These excitatory local neurons are a self-reinforcing clique. Note too `MeVC1 → Dm17` (optic lobe) and the Kenyon cell outputs, which are one-way (driven, not looping).

In [ ]:
r_late, _ = simulate_brain(W_sens, stim_25, duration_ms=400, weight_scale=0.581,
                           stim_off_ms=100, count_after_ms=200, verbose=False)
late = r_late > 0
late_idx = np.flatnonzero(late)
print(f"Neurons still firing 100+ ms after the stimulus: {len(late_idx):,}")

sub = W_sens[late][:, late].tocoo()
pairs = pd.DataFrame({
    "pre": late_idx[sub.row],
    "post": late_idx[sub.col],
    "synapses": sub.data,
})
pairs = pairs[pairs["synapses"] > 0]
pairs["pre_type"] = neurons["type"].to_numpy()[pairs["pre"]]
pairs["post_type"] = neurons["type"].to_numpy()[pairs["post"]]

print("\nStrongest excitatory connections within the late-active group, by type:")
(pairs.groupby(["pre_type", "post_type"])["synapses"]
      .sum()
      .sort_values(ascending=False)
      .head(25))

## E9. Knock-out test

Neuroscientists find what a circuit needs by silencing parts of it. `knock_out` zeros a group's outgoing connections.

**Silencing lLN1_bc** cut lingering activity from ~44,000 to 25,690 spikes, and improved selectivity (DA1 75 vs other PNs 60 Hz). An important piece, but not all.

In [ ]:
def knock_out(W, mask):
    return (sp.diags((~mask).astype(np.float32)) @ W).tocsr()

is_lLN1_bc = (results["type"] == "lLN1_bc").to_numpy()
W_ko = knock_out(W_sens, is_lLN1_bc)

r_on, _ = simulate_brain(W_ko, stim_25, duration_ms=300, weight_scale=0.581, verbose=False)
_, act = simulate_brain(W_ko, stim_25, duration_ms=400, weight_scale=0.581,
                        stim_off_ms=100, verbose=False)
per_ms = act.reshape(-1, 10).sum(axis=1)
print(f"Spikes after stimulus: {per_ms[300:].sum():,}")
print(f"DA1 PN rate: {r_on[is_da1].mean():.1f}   other PN rate: {r_on[is_other_pn].mean():.1f}")

## E10. Are these local neurons really excitatory?

**Yes:** `lLN1_bc`, `lLN2T_a` and `lLN2T_c` are **lab-verified as acetylcholine**. Real flies do have **excitatory local neurons** that spread activity between glomeruli, so we shouldn't flip their sign.

In [ ]:
nt = pd.read_feather("../data/body-neurotransmitters-male-cns-v1.0.feather",
                     columns=["cell_type", "consensus_nt", "celltype_predicted_nt",
                              "celltype_predicted_nt_confidence", "ground_truth"])

suspects = ["lLN1_bc", "lLN2X12", "lLN2T_c", "lLN2T_a", "lLN2F_a"]
(nt[nt["cell_type"].isin(suspects)]
   .groupby("cell_type")
   .agg(consensus=("consensus_nt", "first"),
        type_prediction=("celltype_predicted_nt", "first"),
        type_confidence=("celltype_predicted_nt_confidence", "mean"),
        lab_verified=("ground_truth", lambda s: s.dropna().unique().tolist())))

## E11. Scan every cell type for self-excitation

For each type, count excitatory synapses among its own members, per neuron.

**`lLN1_bc` is an extreme outlier**: 2,895 per neuron, three times the next type.

**Not every loop is a bug.** `PFNv`, `PEN_a`, `PEN_b` and `FR1` are **central complex** neurons. With `EPG` neurons they form a **ring attractor**, the fly's internal compass: a "bump" of activity that represents heading and **is designed to sustain itself**, so the fly remembers its direction in darkness. Persistence there is real biology.

In [ ]:
coo = W_sens.tocoo()
types = neurons["type"].to_numpy()
has_type = pd.notna(types[coo.row]) & pd.notna(types[coo.col])
same_type = has_type & (types[coo.row] == types[coo.col])
mask = same_type & (coo.data > 0) & (coo.row != coo.col)

within = pd.DataFrame({"type": types[coo.row[mask]], "synapses": coo.data[mask]})
scan = within.groupby("type")["synapses"].sum().to_frame()
scan["neurons"] = neurons["type"].value_counts().reindex(scan.index).to_numpy()
scan["synapses_per_neuron"] = scan["synapses"] / scan["neurons"]
scan.sort_values("synapses_per_neuron", ascending=False).head(20)

## E12. Remove only the lLN1_bc clique

**Barely helped** (44,000 → 42,760), and all 30 lLN1_bc are *still* firing. They're kept alive by other neurons, not just each other.

In [ ]:
coo = W_sens.tocoo()
drop = is_lLN1_bc[coo.row] & is_lLN1_bc[coo.col]
W_cal = sp.csr_matrix((coo.data[~drop], (coo.row[~drop], coo.col[~drop])), shape=W_sens.shape)
print("Removed lLN1_bc → lLN1_bc connections:", drop.sum())

r_on, _ = simulate_brain(W_cal, stim_25, duration_ms=300, weight_scale=0.581, verbose=False)
r_late, act = simulate_brain(W_cal, stim_25, duration_ms=400, weight_scale=0.581,
                             stim_off_ms=100, count_after_ms=200, verbose=False)
per_ms = act.reshape(-1, 10).sum(axis=1)

print(f"Spikes after stimulus: {per_ms[300:].sum():,}")
print(f"DA1 PN rate: {r_on[is_da1].mean():.1f}   other PN rate: {r_on[is_other_pn].mean():.1f}")
print(f"Neurons ≥50 Hz during stimulus: {(r_on >= 50).sum() - len(stim_25):,}")

late = results.assign(late_rate=r_late)
late = late[late["late_rate"] > 0]
print(f"\nNeurons still firing 100+ ms after the stimulus: {len(late):,}")
print(late.groupby("superclass").size().sort_values(ascending=False).head(8))
print()
late["type"].value_counts().head(20)

## E13. Rule: no excitation among antennal lobe local neurons

Instead of one type at a time, apply one rule to the whole class `ALLN`: remove **excitatory** local-neuron → local-neuron connections. Their excitation of PNs and their inhibition all stay.

**Selectivity jumps:** DA1 PNs **193 Hz vs other PNs 101 Hz**, nearly 2:1. The web of excitatory local neurons was what blurred the pheromone signal. But persistence didn't change.

In [ ]:
is_alln = (neurons["class"] == "ALLN").to_numpy()
print("Antennal lobe local neurons:", is_alln.sum())

coo = W_cal.tocoo()
drop = is_alln[coo.row] & is_alln[coo.col] & (coo.data > 0)
W_cal2 = sp.csr_matrix((coo.data[~drop], (coo.row[~drop], coo.col[~drop])), shape=W_cal.shape)
print(f"Removed {drop.sum():,} excitatory ALLN → ALLN connections "
      f"({coo.data[drop].sum():,.0f} synapses)")

r_on, _ = simulate_brain(W_cal2, stim_25, duration_ms=300, weight_scale=0.581, verbose=False)
r_late, act = simulate_brain(W_cal2, stim_25, duration_ms=400, weight_scale=0.581,
                             stim_off_ms=100, count_after_ms=200, verbose=False)
per_ms = act.reshape(-1, 10).sum(axis=1)

print(f"\nSpikes after stimulus: {per_ms[300:].sum():,}")
print(f"DA1 PN rate: {r_on[is_da1].mean():.1f}   other PN rate: {r_on[is_other_pn].mean():.1f}")
print(f"Neurons ≥50 Hz during stimulus: {(r_on >= 50).sum() - len(stim_25):,}")

late = results.assign(late_rate=r_late)
late = late[late["late_rate"] > 0]
print(f"\nNeurons still firing 100+ ms after the stimulus: {len(late):,}")
print(late.groupby("superclass").size().sort_values(ascending=False).head(8))
print()
late["type"].value_counts().head(20)

## E14. What keeps lLN1_bc firing?

Its strongest drivers are **projection neurons from many glomeruli**. So the loop is:
PNs → excite local neurons → excite PNs → ...

In [ ]:
late_mask = (r_late > 0)
inputs = W_cal2[late_mask][:, is_lLN1_bc].tocoo()
late_pos = np.flatnonzero(late_mask)

drivers = pd.DataFrame({
    "pre_type": neurons["type"].to_numpy()[late_pos[inputs.row]],
    "synapses": inputs.data,
})
drivers = drivers[drivers["synapses"] > 0]
drivers.groupby("pre_type")["synapses"].sum().sort_values(ascending=False).head(15)

## E15. A general biological rule: cap each neuron's excitation

Real neurons regulate their own sensitivity: a neuron receiving far more excitation than usual turns its excitatory synapses down. This is called **homeostatic synaptic scaling**. Our model has no such brake.

We approximate it: compute each neuron's total excitatory input; any neuron above the **95th percentile** (the top 5%) gets all its excitatory inputs scaled down to the cap. Inhibition is untouched.

The function works in **CSC** format (columns grouped together), because a column is a neuron's incoming connections.

**Cap: 1,819 synapses. 7,428 neurons scaled down; lLN1_bc to 53%.**

In [ ]:
def cap_excitation(W, percentile=95):
    Wc = W.tocsc(copy=True)
    pos = Wc.copy()
    pos.data = np.clip(pos.data, 0, None)
    exc_in = np.asarray(pos.sum(axis=0)).ravel()

    cap = np.percentile(exc_in[exc_in > 0], percentile)
    factor = np.minimum(1.0, cap / np.maximum(exc_in, 1))

    column_of_entry = np.repeat(np.arange(W.shape[1]), np.diff(Wc.indptr))
    Wc.data = np.where(Wc.data > 0, Wc.data * factor[column_of_entry], Wc.data)
    return Wc.tocsr(), cap, factor

W_capped, cap, factor = cap_excitation(W_cal2, percentile=95)
print(f"Cap: {cap:,.0f} excitatory synapses per neuron")
print(f"Neurons scaled down: {(factor < 1).sum():,}")
print(f"lLN1_bc scaled to: {factor[is_lLN1_bc].mean():.2f} of original")

## E16. Retest

**Spikes after stimulus: 0.** The network calms down on its own.
**DA1 PNs 43.8 Hz, other PNs 0.1 Hz:** near-perfect selectivity.

One general, biologically motivated rule did what several targeted patches couldn't. But only 5 neurons fire at 50 Hz or more: maybe now *too* quiet.

In [ ]:
r_on, _ = simulate_brain(W_capped, stim_25, duration_ms=300, weight_scale=0.581, verbose=False)
r_late, act = simulate_brain(W_capped, stim_25, duration_ms=400, weight_scale=0.581,
                             stim_off_ms=100, count_after_ms=200, verbose=False)
per_ms = act.reshape(-1, 10).sum(axis=1)

print(f"Spikes after stimulus: {per_ms[300:].sum():,}")
print(f"DA1 PN rate: {r_on[is_da1].mean():.1f}   other PN rate: {r_on[is_other_pn].mean():.1f}")
print(f"Neurons ≥50 Hz during stimulus: {(r_on >= 50).sum() - len(stim_25):,}")

late = results.assign(late_rate=r_late)
late = late[late["late_rate"] > 0]
print(f"\nNeurons still firing 100+ ms after the stimulus: {len(late):,}")
late["type"].value_counts().head(15)

## E17. Choosing the cap

Test caps at the 95th, 97th and 99th percentiles, with weak (25 ORNs) and strong (204) stimuli, and track how far the signal travels: lateral horn neurons (`LH...`, innate smell responses) and descending neurons (commands to the body).

- **95 and 97 are stable** (0 lingering spikes); **99 runs away**.
- Responses are **graded**: stronger smell → stronger, wider response.
- At 97, 25 ORNs reach 31 lateral horn neurons and 5 descending neurons.

In [ ]:
is_lh = results["type"].str.startswith("LH", na=False).to_numpy()
is_dn = (results["superclass"] == "descending_neuron").to_numpy()

rows = []
for pct in [95, 97, 99]:
    W_test, cap_value, f = cap_excitation(W_cal2, percentile=pct)
    _, act = simulate_brain(W_test, stim_25, duration_ms=400, weight_scale=0.581,
                            stim_off_ms=100, verbose=False)
    after = int(act.reshape(-1, 10).sum(axis=1)[300:].sum())

    for k in [25, 204]:
        stim_k = np.random.default_rng(42).choice(all_orns, size=k, replace=False)
        r, _ = simulate_brain(W_test, stim_k, duration_ms=300, weight_scale=0.581, verbose=False)
        rows.append({
            "percentile": pct,
            "ORNs": k,
            "spikes after stim (25 ORNs)": after,
            "DA1 PN": round(r[is_da1].mean(), 1),
            "other PN": round(r[is_other_pn].mean(), 1),
            "neurons >10 Hz": int((r > 10).sum()) - k,
            "LH neurons >10 Hz": int((r[is_lh] > 10).sum()),
            "DNs >10 Hz": int((r[is_dn] > 10).sum()),
        })
    print(f"percentile {pct} done", flush=True)

pd.DataFrame(rows)

## E18. Save the calibrated brain (first version)

We save the 97-percentile matrix and a JSON record of every decision. Writing down *how* a dataset was changed is good scientific practice.

(Part F replaces this with the 95 version.)

In [ ]:
import json

W_final, cap_final, _ = cap_excitation(W_cal2, percentile=97)
sp.save_npz("../data/W_calibrated.npz", W_final)

calibration = {
    "dataset": "MaleCNS v1.0, minconf 0.5",
    "weight_scale": 0.581,
    "excitation_cap_percentile": 97,
    "excitation_cap_synapses": float(cap_final),
    "changes": [
        "histamine treated as inhibitory",
        "dopamine, octopamine, serotonin outputs removed (slow modulators)",
        "Kenyon cell to Kenyon cell connections removed",
        "brain input onto sensory neurons removed",
        "lLN1_bc to lLN1_bc connections removed",
        "excitatory ALLN to ALLN connections removed",
        "each neuron's total excitatory input capped at the 97th percentile",
    ],
}
with open("../data/calibration.json", "w") as f:
    json.dump(calibration, f, indent=2)
print("Saved W_calibrated.npz and calibration.json")

# Part F: Can smell steer the fly?

## F1. Find the neurons

For an odor-tracking fly, we use an attractive food odor: **vinegar**, detected by the **DM1** and **VA2** glomeruli (74 and 83 ORNs).

**DNa02** is a descending neuron known to be involved in steering: one on each side. If smell on the left made left DNa02 fire more, the fly could turn toward food.

In [ ]:
W_final = sp.load_npz("../data/W_calibrated.npz")

print(find_types("ORN_DM1"))
print(find_types("ORN_VA2"))
print(neurons[neurons["type"] == "DNa02"][["bodyId", "type", "somaSide"]])

## F2. Stimulate one antenna at a time

ORNs' cell bodies are in the antenna, outside the brain, so we use **`rootSide`** to know which antenna each belongs to.

**DNa02 stays at 0 Hz.** The smell signal doesn't reach it. Steering is driven by many inputs (vision, navigation circuits), and smell may use other descending neurons.

In [ ]:
vinegar = neurons["type"].isin(["ORN_DM1", "ORN_VA2"])
left_orns = np.flatnonzero(vinegar & (neurons["rootSide"] == "L"))
right_orns = np.flatnonzero(vinegar & (neurons["rootSide"] == "R"))
print(f"Vinegar ORNs: {len(left_orns)} left, {len(right_orns)} right")

dna02_L = np.flatnonzero((neurons["type"] == "DNa02") & (neurons["somaSide"] == "L"))
dna02_R = np.flatnonzero((neurons["type"] == "DNa02") & (neurons["somaSide"] == "R"))

for label, stim in [("left antenna", left_orns), ("right antenna", right_orns)]:
    r, _ = simulate_brain(W_final, stim, duration_ms=500, weight_scale=0.581, verbose=False)
    print(f"{label}: DNa02 left {r[dna02_L].mean():.1f} Hz, right {r[dna02_R].mean():.1f} Hz, "
          f"neurons >10 Hz: {(r > 10).sum() - len(stim)}")

## F3. Stability check with vinegar

**30,032 lingering spikes:** with a strong vinegar stimulus, the 97 cap tips into runaway. 97 was close to the tipping point, and a different odor pushed it over.

In [ ]:
rates_by_side = {}
for side, stim in [("L", left_orns), ("R", right_orns)]:
    r, _ = simulate_brain(W_final, stim, duration_ms=500, weight_scale=0.581, verbose=False)
    rates_by_side[side] = r
    print(f"stimulated {side} done", flush=True)

_, act = simulate_brain(W_final, left_orns, duration_ms=400, weight_scale=0.581,
                        stim_off_ms=100, verbose=False)
print("Spikes after stimulus (vinegar):", int(act.reshape(-1, 10).sum(axis=1)[300:].sum()))

## F4. Which descending neurons flip with the smell's side?

`unstack` puts left and right neurons side by side. For each type:
- difference (left − right) with smell on the left, and with smell on the right;
- **flip** = the change between them. A good steering neuron's difference **flips** sign.

The flips are small (≤14 Hz), many come from neurons firing on one side only, and rates come in 2 Hz steps (one spike in 500 ms), so much of this is noise.

In [ ]:
dn = neurons[(neurons["superclass"] == "descending_neuron") &
             (neurons["somaSide"].isin(["L", "R"]))][["type", "somaSide"]].copy()
dn["smell_left"] = rates_by_side["L"][dn.index]
dn["smell_right"] = rates_by_side["R"][dn.index]

table = dn.groupby(["type", "somaSide"])[["smell_left", "smell_right"]].mean().unstack("somaSide")
table = table.dropna()

diff_when_left = table[("smell_left", "L")] - table[("smell_left", "R")]
diff_when_right = table[("smell_right", "L")] - table[("smell_right", "R")]
table["flip"] = diff_when_left - diff_when_right

table.sort_values("flip", key=abs, ascending=False).head(15).round(1)

## F5. Find a stable setting

**The 95 cap is stable at both smell strengths; 97 is not.** 95 becomes the setting.

In [ ]:
# Find a stable setting for vinegar
W95, cap95, _ = cap_excitation(W_cal2, percentile=95)

for name, W_test in [("97", W_final), ("95", W95)]:
    for rate in [150, 50]:
        _, act = simulate_brain(W_test, left_orns, duration_ms=400, weight_scale=0.581,
                                stim_rate=rate, stim_off_ms=100, verbose=False)
        after = int(act.reshape(-1, 10).sum(axis=1)[300:].sum())
        print(f"cap {name}, ORN rate {rate} Hz: spikes after stimulus = {after:,}")

## F6. A careful scan, averaged over three runs

Averaging several random seeds reduces noise.

**At a 50 Hz smell, every descending neuron is silent (all zeros).** The cap that stops runaway also weakens how far signals travel.

In [ ]:
W_use = W95
rate_use = 50

def dn_flip_scan(W_test, rate, seeds=(0, 1, 2)):
    avg = {}
    for side, stim in [("L", left_orns), ("R", right_orns)]:
        runs = [simulate_brain(W_test, stim, duration_ms=500, seed=s, weight_scale=0.581,
                               stim_rate=rate, verbose=False)[0] for s in seeds]
        avg[side] = np.mean(runs, axis=0)
        print(f"stimulated {side} done", flush=True)

    dn = neurons[(neurons["superclass"] == "descending_neuron") &
                 (neurons["somaSide"].isin(["L", "R"]))][["type", "somaSide"]].copy()
    dn["smell_left"] = avg["L"][dn.index]
    dn["smell_right"] = avg["R"][dn.index]

    table = dn.groupby(["type", "somaSide"])[["smell_left", "smell_right"]].mean().unstack("somaSide").dropna()
    diff_when_left = table[("smell_left", "L")] - table[("smell_left", "R")]
    diff_when_right = table[("smell_right", "L")] - table[("smell_right", "R")]
    table["flip"] = diff_when_left - diff_when_right
    table["total"] = table[[("smell_left", "L"), ("smell_left", "R"),
                            ("smell_right", "L"), ("smell_right", "R")]].mean(axis=1)
    return table

scan = dn_flip_scan(W_use, rate_use)
print("\nStrongest left-right flips:")
print(scan.sort_values("flip", key=abs, ascending=False).head(10).round(1))
print("\nStrongest overall responses to vinegar:")
print(scan.sort_values("total", ascending=False).head(10).round(1))

## F7. Where does the signal stop?

Follow the vinegar signal stage by stage:
- **50 Hz:** vinegar PNs respond strongly (55 Hz) and selectively, but it fades in the lateral horn (8 neurons), and no descending neurons.
- **150 Hz:** it travels all the way: 351 lateral horn neurons and 11 descending neurons.

In a real odor plume, smell is faint far away and strong up close, so a fly would mostly "notice" vinegar near the food.

In [ ]:
is_vinegar_pn = results["type"].str.startswith(("DM1_", "VA2_"), na=False).to_numpy()

for rate in [50, 150]:
    r, _ = simulate_brain(W95, left_orns, duration_ms=500, weight_scale=0.581,
                          stim_rate=rate, verbose=False)
    print(f"ORN rate {rate} Hz: vinegar PNs {r[is_vinegar_pn].mean():.1f} Hz, "
          f"other PNs {r[is_other_pn & ~is_vinegar_pn].mean():.1f} Hz, "
          f"LH neurons >10 Hz: {(r[is_lh] > 10).sum()}, "
          f"DNs >10 Hz: {(r[is_dn] > 10).sum()}, "
          f"all neurons >10 Hz: {(r > 10).sum() - len(left_orns)}")

## F8. Rescan with the strong smell

Now there are real candidates. Example, **DNpe002**: the left neuron always fires more (a fixed **bias**), *plus* a steering signal on top: moving the smell from left to right raises the left neuron and lowers the right one. `DNc02` does the reverse.

In [ ]:
scan150 = dn_flip_scan(W95, 150)
print("\nStrongest left-right flips:")
print(scan150.sort_values("flip", key=abs, ascending=False).head(10).round(1))
print("\nStrongest overall responses to vinegar:")
print(scan150.sort_values("total", ascending=False).head(10).round(1))

## F9. Save the 95-percentile brain

In [ ]:
sp.save_npz("../data/W_calibrated.npz", W95)

with open("../data/calibration.json") as f:
    calibration = json.load(f)
calibration["excitation_cap_percentile"] = 95
calibration["excitation_cap_synapses"] = float(cap95)
calibration["changes"][-1] = "each neuron's total excitatory input capped at the 95th percentile"
with open("../data/calibration.json", "w") as f:
    json.dump(calibration, f, indent=2)
print("Saved the 95-percentile calibrated brain")

## F10. Build a steering readout

For each of five types:
1. left rate − right rate,
2. minus that type's **bias** (the midpoint between the two experiments, like zeroing a scale),
3. times a **direction** (+1 or −1) so every type "votes" the same way.

Positive total = turn right; negative = turn left.

In [ ]:
steer_types = ["DNpe002", "DNp56", "DNbe007", "DNg56", "DNc02"]
s = scan150.loc[steer_types]

diff_left = s[("smell_left", "L")] - s[("smell_left", "R")]
diff_right = s[("smell_right", "L")] - s[("smell_right", "R")]
bias = (diff_left + diff_right) / 2
direction = -np.sign(s["flip"])

steering = {
    "measured_with": "W_calibrated (95 cap), vinegar ORNs (DM1, VA2) at 150 Hz, 3 seeds",
    "types": steer_types,
    "direction": {t: float(direction[t]) for t in steer_types},
    "bias_hz": {t: float(bias[t]) for t in steer_types},
}
with open("../data/steering.json", "w") as f:
    json.dump(steering, f, indent=2)

print(pd.DataFrame({"bias (Hz)": bias, "direction": direction}).round(1))

## F11. Save each type's typical activity

Far from food, the steering neurons are almost silent, and subtracting a fixed 23 Hz bias from silence would create a fake turn. Saving each type's typical activity lets the readout scale its bias by how active the neurons currently are.

The `FutureWarning` is harmless: `scan150.loc[t, "total"]` returns a one-value Series. Writing `float(scan150.loc[t, "total"].iloc[0])` silences it.

In [ ]:
with open("../data/steering.json") as f:
    steering = json.load(f)
steering["total_hz"] = {t: float(scan150.loc[t, "total"]) for t in steer_types}
with open("../data/steering.json", "w") as f:
    json.dump(steering, f, indent=2)
print(steering["total_hz"])

# Epilogue: what happened next

These steps happened in Python scripts (`sims/`), not in this notebook.

1. **Steering failed in the arena.** The readout was calibrated with one antenna fully on and the other off. In the arena, both antennae smell similar amounts (about 3 Hz apart), and the descending neurons barely noticed the difference. Each antenna wires into *both* sides of the brain, so left-right differences are small by design.
2. **The brain got stuck "on".** A test with a **1-second** stimulus (instead of 100 ms) revealed another self-sustaining state. The loop: **excitatory local neurons ↔ projection neurons**. Real excitatory local neurons excite PNs mainly through **electrical synapses**, which aren't in the connectome. So two more rules were added: no excitatory local neuron → PN, and no excitatory PN → PN chemical connections.
3. **Calibration became code.** All rules now live in `flybrain/calibrate.py` (`build_calibrated`), and `calibration.json` records them.
4. **The fly navigates with its antennal lobe.** The vinegar PNs give a clean, graded, selective signal, and a **run-and-tumble** rule (like bacteria use) turns its changes into movement. Brain-guided flies spent about 3× more time near the food than brainless controls.
5. **625 neurons are enough.** Only 625 neurons ever fire in response to vinegar. That reduced circuit is spike-for-spike identical to the full brain for these inputs, and runs live in the web app.

## Lessons from this notebook

- **A wiring diagram plus simple rules can break in revealing ways.** Each loop pointed to something the point-neuron model can't represent: axo-axonal synapses, slow modulators, electrical synapses.
- **Test strategies matter.** Switch-off tests, dose-response curves, knock-outs and "who's still firing late" each found something the others missed.
- **Not all persistence is a bug.** The fly's compass is *meant* to sustain itself.
- **Write down every decision.** `calibration.json` lets anyone see exactly how the simulated brain differs from the raw connectome.